Step 1 — Create the database and load the data

In [1]:
import pandas as pd
import sqlite3
import os

# Load cleaned data
df = pd.read_csv('data/diabetes_cleaned.csv')

# Create SQLite database
conn = sqlite3.connect('data/diabetes.db')

# Write dataframe into database as a table called 'patients'
df.to_sql('patients', conn, if_exists='replace', index=False)

print("Database created!")
print(f"Rows loaded: {len(df)}")
print(f"\nColumns in database:")
for col in df.columns:
    print(f"  {col}")

Database created!
Rows loaded: 768

Columns in database:
  Pregnancies
  Glucose
  BloodPressure
  SkinThickness
  Insulin
  BMI
  DiabetesPedigreeFunction
  Age
  Outcome
  BMI_Category
  Age_Group
  Glucose_Risk


Step 2 — Query 1: Overall diabetes statistics

In [2]:
query1 = """
SELECT
    COUNT(*)                                        AS total_patients,
    SUM(Outcome)                                    AS diabetic_patients,
    ROUND(100.0 * SUM(Outcome) / COUNT(*), 1)       AS diabetic_pct,
    ROUND(AVG(Glucose), 1)                          AS avg_glucose,
    ROUND(AVG(BMI), 1)                              AS avg_bmi,
    ROUND(AVG(Age), 1)                              AS avg_age
FROM patients
"""

result1 = pd.read_sql_query(query1, conn)
print("=== Overall Statistics ===")
print(result1.to_string(index=False))

=== Overall Statistics ===
 total_patients  diabetic_patients  diabetic_pct  avg_glucose  avg_bmi  avg_age
            768                268          34.9        121.7     32.5     33.2


Step 3 — Query 2: Risk breakdown by BMI category

In [3]:
query2 = """
SELECT
    BMI_Category,
    COUNT(*)                                        AS total,
    SUM(Outcome)                                    AS diabetic,
    ROUND(100.0 * SUM(Outcome) / COUNT(*), 1)       AS diabetes_rate_pct,
    ROUND(AVG(Glucose), 1)                          AS avg_glucose,
    ROUND(AVG(Age), 1)                              AS avg_age
FROM patients
GROUP BY BMI_Category
ORDER BY diabetes_rate_pct DESC
"""

result2 = pd.read_sql_query(query2, conn)
print("\n=== Diabetes Rate by BMI Category ===")
print(result2.to_string(index=False))


=== Diabetes Rate by BMI Category ===
BMI_Category  total  diabetic  diabetes_rate_pct  avg_glucose  avg_age
       Obese    483       221               45.8        126.2     33.7
  Overweight    179        40               22.3        117.0     32.9
      Normal    102         7                6.9        109.1     31.9
 Underweight      4         0                0.0         95.3     24.0


Step 4 — Query 3: Risk breakdown by glucose level

In [4]:
query3 = """
SELECT
    Glucose_Risk,
    COUNT(*)                                        AS total_patients,
    SUM(Outcome)                                    AS diabetic,
    ROUND(100.0 * SUM(Outcome) / COUNT(*), 1)       AS diabetes_rate_pct,
    ROUND(AVG(BMI), 1)                              AS avg_bmi,
    ROUND(AVG(Age), 1)                              AS avg_age
FROM patients
GROUP BY Glucose_Risk
ORDER BY diabetes_rate_pct DESC
"""

result3 = pd.read_sql_query(query3, conn)
print("\n=== Diabetes Rate by Glucose Risk Level ===")
print(result3.to_string(index=False))


=== Diabetes Rate by Glucose Risk Level ===
Glucose_Risk  total_patients  diabetic  diabetes_rate_pct  avg_bmi  avg_age
    Diabetic             297       176               59.3     34.3     37.0
Pre-diabetic             279        78               28.0     31.9     32.1
      Normal             192        14                7.3     30.4     29.2


Step 5 — Query 4: High risk patient segments

In [5]:
query4 = """
SELECT
    Age_Group,
    BMI_Category,
    COUNT(*)                                        AS total,
    SUM(Outcome)                                    AS diabetic,
    ROUND(100.0 * SUM(Outcome) / COUNT(*), 1)       AS diabetes_rate_pct,
    ROUND(AVG(Glucose), 1)                          AS avg_glucose
FROM patients
GROUP BY Age_Group, BMI_Category
HAVING total >= 10
ORDER BY diabetes_rate_pct DESC
LIMIT 10
"""

result4 = pd.read_sql_query(query4, conn)
print("\n=== Top 10 Highest Risk Patient Segments ===")
print(result4.to_string(index=False))


=== Top 10 Highest Risk Patient Segments ===
Age_Group BMI_Category  total  diabetic  diabetes_rate_pct  avg_glucose
   Senior        Obese     57        37               64.9        137.2
   Middle        Obese    168       100               59.5        130.1
   Senior   Overweight     18        10               55.6        131.4
  Elderly        Obese     12         6               50.0        141.8
   Middle   Overweight     47        18               38.3        119.8
    Young        Obese    246        78               31.7        120.3
   Senior       Normal     16         4               25.0        127.8
   Middle       Normal     18         2               11.1        114.6
    Young   Overweight    102        11               10.8        110.7
  Elderly   Overweight     12         1                8.3        137.8


This is the most impressive query — it finds which combination of age group and BMI category has the highest diabetes rate. This is the kind of insight a healthcare analyst would present to a clinical team.

Step 6 — Save results to CSV

In [6]:
result1.to_csv('data/sql_overall_stats.csv',     index=False)
result2.to_csv('data/sql_bmi_breakdown.csv',      index=False)
result3.to_csv('data/sql_glucose_breakdown.csv',  index=False)
result4.to_csv('data/sql_risk_segments.csv',      index=False)

conn.close()
print("\nAll results saved to data/ folder!")


All results saved to data/ folder!
